# 01 IO Semantic Alignment

`OctoSense.io` is the sample-level module in OctoSense's
`Unified Data Abstraction`. It solves the problem that wireless
sensing files come from different devices and carry different axis
layouts and metadata conventions. The design turns each raw capture
into a shared `RadioTensor` with explicit RF semantics, so later
code can work with semantic labels instead of device-specific file
details. This notebook shows that result by reading one IWL5300
sample and one ESP32 sample, then checking that both outputs expose
the same downstream structure.

### Set Up The Public Reader Workspace

Load the public WiFi reader classes and the notebook-local path
setup so the walkthrough can move directly into semantic inspection
with deterministic repository-relative paths.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = (NOTEBOOK_DIR / "../..").resolve()
SRC_ROOT = REPO_ROOT / "src"
TEST_DATA_ROOT = NOTEBOOK_DIR / "data"
PRESET_ROOT = NOTEBOOK_DIR / "presets"
NOTEBOOK_TREE_DEPTH = 2
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from loguru import logger
import octosense as octo
from octosense.io.readers.wifi import ESP32Reader, IWL5300Reader


logger.remove()
logger.add(sys.stderr, colorize=False, format="{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}")
logger.info("Notebook setup completed with repository-relative paths")
logger.info("Default describe tree render depth set to {}", NOTEBOOK_TREE_DEPTH)

### Read Representative WiFi Samples Through A Shared Semantic Contract

Read one IWL5300 capture and one ESP32 capture, print their semantic
summaries, and verify that both public readers populate the same
axis schema and the required metadata scalars for downstream use.

In [ ]:
logger.info("Reading representative WiFi formats through public reader classes")
iwl_sample_path = TEST_DATA_ROOT / "iwl5300" / "user1-1-1-1-1-r1.dat"
esp_sample_path = TEST_DATA_ROOT / "esp32" / "esp32_c5_test.csv"
if not iwl_sample_path.exists():
    raise FileNotFoundError(
        f"Missing notebook-local IWL5300 sample at {iwl_sample_path}. "
        "Populate demo/notebook/data/iwl5300 before running 01_io.ipynb."
    )
if not esp_sample_path.exists():
    raise FileNotFoundError(
        f"Missing notebook-local ESP32 sample at {esp_sample_path}. "
        "Populate demo/notebook/data/esp32 before running 01_io.ipynb."
    )
iwl_reader = IWL5300Reader(channel=36)
esp_reader = ESP32Reader()

iwl_sample = iwl_reader.read(iwl_sample_path)
esp_sample = esp_reader.read(esp_sample_path)
print(iwl_sample.describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
print(esp_sample.describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
assert iwl_sample.axis_schema.axes == esp_sample.axis_schema.axes
assert iwl_sample.metadata.center_freq is not None
assert iwl_sample.metadata.bandwidth is not None
assert iwl_sample.metadata.sample_rate is not None
assert esp_sample.metadata.center_freq is not None
assert esp_sample.metadata.bandwidth is not None
assert esp_sample.metadata.sample_rate is not None
logger.info("IWL5300 axes: {}", iwl_sample.axis_schema.axes)
logger.info("ESP32 axes: {}", esp_sample.axis_schema.axes)
logger.info(
    "IWL5300 sample path uses read(); center_freq={} bandwidth={} sample_rate={} (center_freq declared via channel=36, bandwidth from packet header, sample_rate from timestamp_low deltas)",
    iwl_sample.metadata.center_freq,
    iwl_sample.metadata.bandwidth,
    iwl_sample.metadata.sample_rate,
)
logger.info(
    "ESP32 sample path uses read(); center_freq={} bandwidth={} sample_rate={} (center_freq from CSV channel, bandwidth from CSI shape, sample_rate from local_timestamp deltas)",
    esp_sample.metadata.center_freq,
    esp_sample.metadata.bandwidth,
    esp_sample.metadata.sample_rate,
)
logger.info(
    "Mandatory readers align on the same semantic axes: {}",
    iwl_sample.axis_schema.axes == esp_sample.axis_schema.axes,
)
logger.info("Shared downstream metadata checks passed for both public readers")